<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/plasma%20proteomic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# StrokeCog: Fast unscaled solver for remaining model only
# Loads CSV files first
# Model: All proteins + Age + Time Since Stroke
# No scaling yet
# ============================================================

import os
import re
import time
import math
import warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import pearsonr, spearmanr
from math import log10

# ------------------------------------------------------------
# 0. Load CSV files
# ------------------------------------------------------------
clinical_file = "Subject_Clinical_Data.csv"
npx_file = "C_NPX_data.csv"

clinical = pd.read_csv(clinical_file)
npx = pd.read_csv(npx_file)

print("Clinical shape:", clinical.shape)
print("NPX shape:", npx.shape)

print("Clinical columns:")
print(clinical.columns.tolist())

print("First few NPX columns:")
print(npx.columns[:10].tolist())

assert "PID" in clinical.columns, "clinical must contain PID"
assert "SIS3" in clinical.columns, "clinical must contain SIS3"
assert "PID" in npx.columns, "npx must contain PID"

# ------------------------------------------------------------
# 1. Prepare NPX data
# ------------------------------------------------------------
# Paper-like ML excludes proteins with any missing NPX values.
NPX_clean = npx.dropna(axis=1).copy()

protein_cols = [c for c in NPX_clean.columns if "OID" in str(c)]

if len(protein_cols) == 0:
    protein_cols = [c for c in NPX_clean.columns if c != "PID"]

print("NPX columns after dropping missing:", NPX_clean.shape[1])
print("Protein features used:", len(protein_cols))

# ------------------------------------------------------------
# 2. Fast unscaled LOOCV function
# ------------------------------------------------------------
def run_fast_unscaled_loocv_model(
    clinical_features,
    model_name,
    max_iter=1000,
    tol=1e-2
):
    needed_cols = ["PID", "SIS3"] + clinical_features

    y_data = clinical[needed_cols].dropna().copy()
    data = y_data.merge(NPX_clean[["PID"] + protein_cols], on="PID", how="inner")

    selected_features = protein_cols + clinical_features

    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", model_name).strip("_")
    pred_file = f"strokecog_predictions_{safe_name}_fast_unscaled_checkpoint.csv"

    if os.path.exists(pred_file):
        pred_table = pd.read_csv(pred_file)
        done_pids = set(pred_table["PID"].values)
        print(f"Resuming from checkpoint: {len(done_pids)} patients already predicted")
    else:
        pred_table = pd.DataFrame(columns=["PID", "SIS3_actual", "SIS3_predicted"])
        done_pids = set()

    pids = list(data["PID"].values)
    start_time = time.time()

    for i, p in enumerate(pids, start=1):

        if p in done_pids:
            continue

        train = data[data["PID"] != p]
        test = data[data["PID"] == p]

        X_train = train[selected_features].values
        y_train = train["SIS3"].values.ravel()

        X_test = test[selected_features].values
        y_test = test["SIS3"].values.ravel()

        model = LogisticRegression(
            fit_intercept=True,
            random_state=42,
            solver="liblinear",
            penalty="l2",
            C=1.0,
            max_iter=max_iter,
            tol=tol
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        new_row = pd.DataFrame({
            "PID": [p],
            "SIS3_actual": [y_test[0]],
            "SIS3_predicted": [y_pred[0]]
        })

        pred_table = pd.concat([pred_table, new_row], ignore_index=True)

        # Save after every patient
        pred_table.to_csv(pred_file, index=False)

        if i % 10 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"Finished {i}/{len(pids)} patients, elapsed {elapsed:.1f} min")

    # --------------------------------------------------------
    # Calculate metrics
    # --------------------------------------------------------
    pred_table = pred_table.sort_values("PID").reset_index(drop=True)

    true_vals = pred_table["SIS3_actual"].values
    pred_vals = pred_table["SIS3_predicted"].values

    rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))

    pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
    spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

    result = {
        "model": model_name,
        "clinical_features": ", ".join(clinical_features),
        "solver": "liblinear",
        "scaling": "no",
        "n_patients": len(true_vals),
        "n_features": len(selected_features),
        "RMSE": rmse,
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "-log10_Pearson_p": -log10(pearson_p),
        "Spearman_r": spearman_r,
        "Spearman_p": spearman_p,
        "-log10_Spearman_p": -log10(spearman_p)
    }

    return result, pred_table

# ------------------------------------------------------------
# 3. Run remaining model only
# ------------------------------------------------------------
model_name = "All proteins + Age + Time Since Stroke"
clinical_features = ["Age", "Time Since Stroke"]

start = time.time()

result, pred_table = run_fast_unscaled_loocv_model(
    clinical_features=clinical_features,
    model_name=model_name,
    max_iter=1000,
    tol=1e-2
)

runtime_min = (time.time() - start) / 60
result["runtime_minutes"] = runtime_min

remaining_result = pd.DataFrame([result])

remaining_result.to_csv("strokecog_remaining_model_fast_unscaled_result.csv", index=False)
pred_table.to_csv("strokecog_remaining_model_fast_unscaled_predictions.csv", index=False)

print("\nFinished remaining model")
print(f"Runtime: {runtime_min:.2f} minutes")
display(remaining_result)

Clinical shape: (85, 14)
NPX shape: (85, 1197)
Clinical columns:
['PID', 'Stroke Size', 'Sex', 'NIHSS', 'Time Since Stroke', 'Age', 'SIS1', 'SIS2', 'SIS3', 'SIS4', 'SIS5', 'SIS6', 'SIS7', 'SIS8']
First few NPX columns:
['OID01272', 'OID01223', 'OID01230', 'OID01216', 'OID01291', 'OID01254', 'OID01275', 'OID01232', 'OID01255', 'OID01258']
NPX columns after dropping missing: 1012
Protein features used: 1011
Finished 10/85 patients, elapsed 0.0 min
Finished 20/85 patients, elapsed 0.1 min
Finished 30/85 patients, elapsed 0.2 min
Finished 40/85 patients, elapsed 0.2 min
Finished 50/85 patients, elapsed 0.3 min
Finished 60/85 patients, elapsed 0.3 min
Finished 70/85 patients, elapsed 0.4 min
Finished 80/85 patients, elapsed 0.4 min


AttributeError: 'numpy.dtypes.ObjectDType' object has no attribute 'dtype'

In [3]:
import pandas as pd
import math
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr, spearmanr
from math import log10

# Load checkpoint prediction file
pred_file = "strokecog_predictions_All_proteins_Age_Time_Since_Stroke_fast_unscaled_checkpoint.csv"

pred_table = pd.read_csv(pred_file)

print(pred_table.dtypes)
print(pred_table.head())

# Convert actual and predicted SIS3 to numeric
pred_table["SIS3_actual"] = pd.to_numeric(pred_table["SIS3_actual"], errors="coerce")
pred_table["SIS3_predicted"] = pd.to_numeric(pred_table["SIS3_predicted"], errors="coerce")

# Drop any problematic rows if present
pred_table = pred_table.dropna(subset=["SIS3_actual", "SIS3_predicted"]).copy()

true_vals = pred_table["SIS3_actual"].astype(float).values
pred_vals = pred_table["SIS3_predicted"].astype(float).values

rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))
pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

remaining_result = pd.DataFrame([{
    "model": "All proteins + Age + Time Since Stroke",
    "clinical_features": "Age, Time Since Stroke",
    "solver": "liblinear",
    "scaling": "no",
    "n_patients": len(true_vals),
    "n_features": 1013,
    "RMSE": rmse,
    "Pearson_r": pearson_r,
    "Pearson_p": pearson_p,
    "-log10_Pearson_p": -log10(pearson_p),
    "Spearman_r": spearman_r,
    "Spearman_p": spearman_p,
    "-log10_Spearman_p": -log10(spearman_p)
}])

remaining_result.to_csv("strokecog_remaining_model_fast_unscaled_result.csv", index=False)
pred_table.to_csv("strokecog_remaining_model_fast_unscaled_predictions_clean.csv", index=False)

display(remaining_result)

PID               int64
SIS3_actual       int64
SIS3_predicted    int64
dtype: object
   PID  SIS3_actual  SIS3_predicted
0    1           89              50
1    3           67              72
2    4           28              72
3    6           92              72
4    8           75              75


,model,clinical_features,solver,scaling,n_patients,n_features,RMSE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p
0,All proteins + Age + Time Since Stroke,"Age, Time Since Stroke",liblinear,no,85,1013,20.269072,0.285623,0.008055,2.093947,0.205935,0.058647,1.231757


In [4]:
# ============================================================
# StrokeCog: Scaled lbfgs LOOCV sensitivity analysis
# 4 models:
# 1. All proteins only
# 2. All proteins + Age
# 3. All proteins + Time Since Stroke
# 4. All proteins + Age + Time Since Stroke
# ============================================================

import os
import re
import time
import math
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import pearsonr, spearmanr
from math import log10

# ------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------
clinical_file = "Subject_Clinical_Data.csv"
npx_file = "C_NPX_data.csv"

clinical = pd.read_csv(clinical_file)
npx = pd.read_csv(npx_file)

print("Clinical shape:", clinical.shape)
print("NPX shape:", npx.shape)

assert "PID" in clinical.columns, "clinical must contain PID"
assert "SIS3" in clinical.columns, "clinical must contain SIS3"
assert "PID" in npx.columns, "npx must contain PID"

# ------------------------------------------------------------
# 1. Prepare NPX data
# ------------------------------------------------------------
# Paper-like ML excludes proteins with any missing NPX values.
NPX_clean = npx.dropna(axis=1).copy()

protein_cols = [c for c in NPX_clean.columns if "OID" in str(c)]

if len(protein_cols) == 0:
    protein_cols = [c for c in NPX_clean.columns if c != "PID"]

print("NPX columns after dropping missing columns:", NPX_clean.shape[1])
print("Protein features used:", len(protein_cols))

# ------------------------------------------------------------
# 2. Helper function
# ------------------------------------------------------------
def safe_neg_log10(p):
    if p <= 0:
        return np.inf
    return -log10(p)


def run_scaled_lbfgs_loocv_model(
    clinical_features,
    model_name,
    max_iter=3000,
    tol=1e-3
):
    """
    Scaled sensitivity analysis:
    - Uses StandardScaler inside each LOOCV fold.
    - Uses LogisticRegression with lbfgs solver.
    - Treats SIS3 as discrete classes, like the paper-like notebook.
    - Saves checkpoint after every patient.
    """

    needed_cols = ["PID", "SIS3"] + clinical_features

    y_data = clinical[needed_cols].dropna().copy()
    data = y_data.merge(NPX_clean[["PID"] + protein_cols], on="PID", how="inner")

    selected_features = protein_cols + clinical_features

    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", model_name).strip("_")
    pred_file = f"strokecog_predictions_{safe_name}_scaled_lbfgs_checkpoint.csv"

    # Resume from checkpoint if available
    if os.path.exists(pred_file):
        pred_table = pd.read_csv(pred_file)
        done_pids = set(pred_table["PID"].astype(str).values)
        print(f"Resuming {model_name}: {len(done_pids)} patients already predicted")
    else:
        pred_table = pd.DataFrame(columns=["PID", "SIS3_actual", "SIS3_predicted"])
        done_pids = set()

    pids = list(data["PID"].values)
    start_time = time.time()

    for i, p in enumerate(pids, start=1):

        if str(p) in done_pids:
            continue

        train = data[data["PID"] != p]
        test = data[data["PID"] == p]

        X_train = train[selected_features].values
        y_train = train["SIS3"].values.ravel()

        X_test = test[selected_features].values
        y_test = test["SIS3"].values.ravel()

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                fit_intercept=True,
                random_state=42,
                solver="lbfgs",
                max_iter=max_iter,
                tol=tol
            )
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        new_row = pd.DataFrame({
            "PID": [p],
            "SIS3_actual": [float(y_test[0])],
            "SIS3_predicted": [float(y_pred[0])]
        })

        pred_table = pd.concat([pred_table, new_row], ignore_index=True)

        # Save after every patient
        pred_table.to_csv(pred_file, index=False)

        if i % 10 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"  {model_name}: finished {i}/{len(pids)} patients, elapsed {elapsed:.1f} min")

    # --------------------------------------------------------
    # Calculate metrics
    # --------------------------------------------------------
    pred_table["SIS3_actual"] = pd.to_numeric(pred_table["SIS3_actual"], errors="coerce")
    pred_table["SIS3_predicted"] = pd.to_numeric(pred_table["SIS3_predicted"], errors="coerce")
    pred_table = pred_table.dropna(subset=["SIS3_actual", "SIS3_predicted"]).copy()

    true_vals = pred_table["SIS3_actual"].astype(float).values
    pred_vals = pred_table["SIS3_predicted"].astype(float).values

    rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))
    pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
    spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

    result = {
        "model": model_name,
        "clinical_features": ", ".join(clinical_features) if clinical_features else "none",
        "solver": "lbfgs",
        "scaling": "yes_inside_LOOCV",
        "n_patients": len(true_vals),
        "n_features": len(selected_features),
        "RMSE": rmse,
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "-log10_Pearson_p": safe_neg_log10(pearson_p),
        "Spearman_r": spearman_r,
        "Spearman_p": spearman_p,
        "-log10_Spearman_p": safe_neg_log10(spearman_p)
    }

    return result, pred_table


# ------------------------------------------------------------
# 3. Run 4 scaled lbfgs models
# ------------------------------------------------------------
model_specs = [
    ([], "All proteins only"),
    (["Age"], "All proteins + Age"),
    (["Time Since Stroke"], "All proteins + Time Since Stroke"),
    (["Age", "Time Since Stroke"], "All proteins + Age + Time Since Stroke")
]

scaled_results = {}
prediction_tables = {}

total_start = time.time()

for clinical_features, model_name in model_specs:

    print("\n====================================================")
    print("Running scaled lbfgs model:", model_name)
    print("Clinical features:", clinical_features if clinical_features else "none")
    print("====================================================")

    start = time.time()

    result, pred_table = run_scaled_lbfgs_loocv_model(
        clinical_features=clinical_features,
        model_name=model_name,
        max_iter=3000,
        tol=1e-3
    )

    runtime_min = (time.time() - start) / 60
    result["runtime_minutes"] = runtime_min

    scaled_results[model_name] = result
    prediction_tables[model_name] = pred_table

    # Save model-specific predictions
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", model_name).strip("_")
    pred_table.to_csv(f"strokecog_predictions_{safe_name}_scaled_lbfgs_final.csv", index=False)

    # Save cumulative results
    scaled_results_df = pd.DataFrame(list(scaled_results.values()))
    scaled_results_df.to_csv("strokecog_scaled_lbfgs_results_partial.csv", index=False)

    print("\nFinished:", model_name)
    print(f"Runtime: {runtime_min:.2f} minutes")
    display(scaled_results_df)

# ------------------------------------------------------------
# 4. Save final scaled results
# ------------------------------------------------------------
scaled_results_df = pd.DataFrame(list(scaled_results.values()))
scaled_results_df.to_csv("strokecog_scaled_lbfgs_results_final.csv", index=False)

total_runtime = (time.time() - total_start) / 60

print("\n====================================================")
print("All scaled lbfgs models finished")
print(f"Total runtime: {total_runtime:.2f} minutes")
print("Saved: strokecog_scaled_lbfgs_results_final.csv")
print("====================================================")

display(scaled_results_df)

Clinical shape: (85, 14)
NPX shape: (85, 1197)
NPX columns after dropping missing columns: 1012
Protein features used: 1011

Running scaled lbfgs model: All proteins only
Clinical features: none


/tmp/ipykernel_1514/2530883930.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pred_table = pd.concat([pred_table, new_row], ignore_index=True)


  All proteins only: finished 10/85 patients, elapsed 0.0 min
  All proteins only: finished 20/85 patients, elapsed 0.1 min
  All proteins only: finished 30/85 patients, elapsed 0.1 min
  All proteins only: finished 40/85 patients, elapsed 0.1 min
  All proteins only: finished 50/85 patients, elapsed 0.2 min
  All proteins only: finished 60/85 patients, elapsed 0.2 min
  All proteins only: finished 70/85 patients, elapsed 0.2 min
  All proteins only: finished 80/85 patients, elapsed 0.3 min

Finished: All proteins only
Runtime: 0.28 minutes


,model,clinical_features,solver,scaling,n_patients,n_features,RMSE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,runtime_minutes
0,All proteins only,none,lbfgs,yes_inside_LOOCV,85,1011,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.276546



Running scaled lbfgs model: All proteins + Age
Clinical features: ['Age']


...

/tmp/ipykernel_1514/2530883930.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pred_table = pd.concat([pred_table, new_row], ignore_index=True)


  All proteins + Age + Time Since Stroke: finished 10/85 patients, elapsed 0.0 min
  All proteins + Age + Time Since Stroke: finished 20/85 patients, elapsed 0.0 min
  All proteins + Age + Time Since Stroke: finished 30/85 patients, elapsed 0.1 min
  All proteins + Age + Time Since Stroke: finished 40/85 patients, elapsed 0.1 min
  All proteins + Age + Time Since Stroke: finished 50/85 patients, elapsed 0.2 min
  All proteins + Age + Time Since Stroke: finished 60/85 patients, elapsed 0.2 min
  All proteins + Age + Time Since Stroke: finished 70/85 patients, elapsed 0.2 min
  All proteins + Age + Time Since Stroke: finished 80/85 patients, elapsed 0.2 min

Finished: All proteins + Age + Time Since Stroke
Runtime: 0.25 minutes


,model,clinical_features,solver,scaling,n_patients,n_features,RMSE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,runtime_minutes
0,All proteins only,none,lbfgs,yes_inside_LOOCV,85,1011,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.276546
1,All proteins + Age,Age,lbfgs,yes_inside_LOOCV,85,1012,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.272804
2,All proteins + Time Since Stroke,Time Since Stroke,lbfgs,yes_inside_LOOCV,85,1012,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.237984
3,All proteins + Age + Time Since Stroke,"Age, Time Since Stroke",lbfgs,yes_inside_LOOCV,85,1013,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.253633



All scaled lbfgs models finished
Total runtime: 1.04 minutes
Saved: strokecog_scaled_lbfgs_results_final.csv


,model,clinical_features,solver,scaling,n_patients,n_features,RMSE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,runtime_minutes
0,All proteins only,none,lbfgs,yes_inside_LOOCV,85,1011,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.276546
1,All proteins + Age,Age,lbfgs,yes_inside_LOOCV,85,1012,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.272804
2,All proteins + Time Since Stroke,Time Since Stroke,lbfgs,yes_inside_LOOCV,85,1012,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.237984
3,All proteins + Age + Time Since Stroke,"Age, Time Since Stroke",lbfgs,yes_inside_LOOCV,85,1013,21.225678,0.150702,0.168604,0.773133,0.057149,0.603407,0.21939,0.253633


In [6]:
import pandas as pd

files = {
    "proteins_only": "strokecog_predictions_All_proteins_only_scaled_lbfgs_final.csv",
    "age": "strokecog_predictions_All_proteins_Age_scaled_lbfgs_final.csv",
    "time": "strokecog_predictions_All_proteins_Time_Since_Stroke_scaled_lbfgs_final.csv",
    "age_time": "strokecog_predictions_All_proteins_Age_Time_Since_Stroke_scaled_lbfgs_final.csv"
}

preds = {}

for name, file in files.items():
    df = pd.read_csv(file)
    df = df[["PID", "SIS3_actual", "SIS3_predicted"]].sort_values("PID")
    preds[name] = df["SIS3_predicted"].values
    print(name, df["SIS3_predicted"].value_counts().sort_index().to_dict())

print("Proteins vs Age same:", (preds["proteins_only"] == preds["age"]).all())
print("Proteins vs Time same:", (preds["proteins_only"] == preds["time"]).all())
print("Proteins vs Age+Time same:", (preds["proteins_only"] == preds["age_time"]).all())

proteins_only {47.0: 3, 50.0: 1, 56.0: 1, 58.0: 2, 64.0: 1, 67.0: 2, 69.0: 1, 72.0: 2, 75.0: 10, 78.0: 2, 81.0: 9, 83.0: 4, 86.0: 5, 89.0: 16, 92.0: 8, 94.0: 9, 97.0: 6, 100.0: 3}
age {47.0: 3, 50.0: 1, 56.0: 1, 58.0: 2, 64.0: 1, 67.0: 2, 69.0: 1, 72.0: 2, 75.0: 10, 78.0: 2, 81.0: 9, 83.0: 4, 86.0: 5, 89.0: 16, 92.0: 8, 94.0: 9, 97.0: 6, 100.0: 3}
time {47.0: 3, 50.0: 1, 56.0: 1, 58.0: 2, 64.0: 1, 67.0: 2, 69.0: 1, 72.0: 2, 75.0: 10, 78.0: 2, 81.0: 9, 83.0: 4, 86.0: 5, 89.0: 16, 92.0: 8, 94.0: 9, 97.0: 6, 100.0: 3}
age_time {47.0: 3, 50.0: 1, 56.0: 1, 58.0: 2, 64.0: 1, 67.0: 2, 69.0: 1, 72.0: 2, 75.0: 10, 78.0: 2, 81.0: 9, 83.0: 4, 86.0: 5, 89.0: 16, 92.0: 8, 94.0: 9, 97.0: 6, 100.0: 3}
Proteins vs Age same: True
Proteins vs Time same: True
Proteins vs Age+Time same: True


In [7]:
# ============================================================
# StrokeCog: Continuous SIS3 model with scaling
# Model type: Ridge regression
# Outcome: SIS3 as continuous mood score
# Scaling: StandardScaler inside each LOOCV fold
# ============================================================

import os
import re
import time
import math
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from math import log10

# ------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------
clinical_file = "Subject_Clinical_Data.csv"
npx_file = "C_NPX_data.csv"

clinical = pd.read_csv(clinical_file)
npx = pd.read_csv(npx_file)

print("Clinical shape:", clinical.shape)
print("NPX shape:", npx.shape)

assert "PID" in clinical.columns
assert "SIS3" in clinical.columns
assert "PID" in npx.columns

# ------------------------------------------------------------
# 1. Prepare NPX data
# ------------------------------------------------------------
# Match paper-like ML: remove protein columns with any missing values.
NPX_clean = npx.dropna(axis=1).copy()

protein_cols = [c for c in NPX_clean.columns if "OID" in str(c)]

if len(protein_cols) == 0:
    protein_cols = [c for c in NPX_clean.columns if c != "PID"]

print("NPX columns after dropping missing:", NPX_clean.shape[1])
print("Protein features used:", len(protein_cols))

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def safe_neg_log10(p):
    if p <= 0:
        return np.inf
    return -log10(p)


def run_scaled_continuous_ridge_loocv(
    clinical_features,
    model_name
):
    """
    Continuous SIS3 sensitivity model:
    - SIS3 is treated as a continuous outcome.
    - Predictors are scaled inside each LOOCV fold.
    - RidgeCV chooses regularization strength using training data only.
    """

    needed_cols = ["PID", "SIS3"] + clinical_features

    y_data = clinical[needed_cols].dropna().copy()
    data = y_data.merge(NPX_clean[["PID"] + protein_cols], on="PID", how="inner")

    selected_features = protein_cols + clinical_features

    pids = list(data["PID"].values)

    true_vals = []
    pred_vals = []
    pid_vals = []
    alpha_vals = []

    # Ridge regularization grid
    alphas = np.logspace(-3, 5, 25)

    start_time = time.time()

    for i, p in enumerate(pids, start=1):

        train = data[data["PID"] != p]
        test = data[data["PID"] == p]

        X_train = train[selected_features].values
        y_train = train["SIS3"].astype(float).values

        X_test = test[selected_features].values
        y_test = test["SIS3"].astype(float).values

        model = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=alphas)
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        true_vals.append(float(y_test[0]))
        pred_vals.append(float(y_pred[0]))
        pid_vals.append(p)

        # Save selected alpha
        ridge_step = model.named_steps["ridgecv"]
        alpha_vals.append(ridge_step.alpha_)

        if i % 10 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"  {model_name}: finished {i}/{len(pids)} patients, elapsed {elapsed:.2f} min")

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    true_vals = np.array(true_vals, dtype=float)
    pred_vals = np.array(pred_vals, dtype=float)

    rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))
    mae = mean_absolute_error(true_vals, pred_vals)

    pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
    spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

    result = {
        "model": model_name,
        "outcome_type": "continuous_SIS3",
        "estimator": "RidgeCV",
        "scaling": "yes_inside_LOOCV",
        "clinical_features": ", ".join(clinical_features) if clinical_features else "none",
        "n_patients": len(true_vals),
        "n_features": len(selected_features),
        "RMSE": rmse,
        "MAE": mae,
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "-log10_Pearson_p": safe_neg_log10(pearson_p),
        "Spearman_r": spearman_r,
        "Spearman_p": spearman_p,
        "-log10_Spearman_p": safe_neg_log10(spearman_p),
        "median_selected_alpha": np.median(alpha_vals),
        "min_selected_alpha": np.min(alpha_vals),
        "max_selected_alpha": np.max(alpha_vals)
    }

    pred_table = pd.DataFrame({
        "PID": pid_vals,
        "SIS3_actual": true_vals,
        "SIS3_predicted_continuous": pred_vals,
        "selected_alpha": alpha_vals
    })

    return result, pred_table


# ------------------------------------------------------------
# 3. Run 4 continuous Ridge models
# ------------------------------------------------------------
model_specs = [
    ([], "All proteins only"),
    (["Age"], "All proteins + Age"),
    (["Time Since Stroke"], "All proteins + Time Since Stroke"),
    (["Age", "Time Since Stroke"], "All proteins + Age + Time Since Stroke")
]

continuous_results = []
continuous_prediction_tables = {}

total_start = time.time()

for clinical_features, model_name in model_specs:

    print("\n====================================================")
    print("Running continuous scaled Ridge model:", model_name)
    print("Clinical features:", clinical_features if clinical_features else "none")
    print("====================================================")

    start = time.time()

    result, pred_table = run_scaled_continuous_ridge_loocv(
        clinical_features=clinical_features,
        model_name=model_name
    )

    runtime_min = (time.time() - start) / 60
    result["runtime_minutes"] = runtime_min

    continuous_results.append(result)
    continuous_prediction_tables[model_name] = pred_table

    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", model_name).strip("_")
    pred_table.to_csv(f"strokecog_predictions_{safe_name}_continuous_scaled_ridge.csv", index=False)

    continuous_results_df = pd.DataFrame(continuous_results)
    continuous_results_df.to_csv("strokecog_continuous_scaled_ridge_results_partial.csv", index=False)

    print("\nFinished:", model_name)
    print(f"Runtime: {runtime_min:.2f} minutes")
    display(continuous_results_df)

# ------------------------------------------------------------
# 4. Save final results
# ------------------------------------------------------------
continuous_results_df = pd.DataFrame(continuous_results)
continuous_results_df.to_csv("strokecog_continuous_scaled_ridge_results_final.csv", index=False)

total_runtime = (time.time() - total_start) / 60

print("\n====================================================")
print("All continuous scaled Ridge models finished")
print(f"Total runtime: {total_runtime:.2f} minutes")
print("Saved: strokecog_continuous_scaled_ridge_results_final.csv")
print("====================================================")

display(continuous_results_df)

Clinical shape: (85, 14)
NPX shape: (85, 1197)
NPX columns after dropping missing: 1012
Protein features used: 1011

Running continuous scaled Ridge model: All proteins only
Clinical features: none
  All proteins only: finished 10/85 patients, elapsed 0.01 min
  All proteins only: finished 20/85 patients, elapsed 0.01 min
  All proteins only: finished 30/85 patients, elapsed 0.01 min
  All proteins only: finished 40/85 patients, elapsed 0.02 min
  All proteins only: finished 50/85 patients, elapsed 0.02 min
  All proteins only: finished 60/85 patients, elapsed 0.03 min
  All proteins only: finished 70/85 patients, elapsed 0.03 min
  All proteins only: finished 80/85 patients, elapsed 0.04 min

Finished: All proteins only
Runtime: 0.04 minutes


,model,outcome_type,estimator,scaling,clinical_features,n_patients,n_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,median_selected_alpha,min_selected_alpha,max_selected_alpha,runtime_minutes
0,All proteins only,continuous_SIS3,RidgeCV,yes_inside_LOOCV,none,85,1011,17.141062,13.733057,0.311306,0.003731,2.428143,0.227991,0.035854,1.445462,1000.0,464.158883,1000.0,0.039732



Running continuous scaled Ridge model: All proteins + Age
Clinical features: ['Age']
  All proteins + Age: finished 10/85 patients, elapsed 0.01 min
  All proteins + Age: finished 20/85 patients, elapsed 0.02 min
  All proteins + Age: finished 30/85 patients, elapsed 0.03 min
  All proteins + Age: finished 40/85 patients, elapsed 0.04 min
  All proteins + Age: finished 50/85 patients, elapsed 0.05 min
  All proteins + Age: finished 60/85 patients, elapsed 0.05 min
  All proteins + Age: finished 70/85 patients, elapsed 0.06 min
  All proteins + Age: finished 80/85 patients, elapsed 0.06 min

Finished: All proteins + Age
Runtime: 0.06 minutes


,model,outcome_type,estimator,scaling,clinical_features,n_patients,n_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,median_selected_alpha,min_selected_alpha,max_selected_alpha,runtime_minutes
0,All proteins only,continuous_SIS3,RidgeCV,yes_inside_LOOCV,none,85,1011,17.141062,13.733057,0.311306,0.003731,2.428143,0.227991,0.035854,1.445462,1000.0,464.158883,1000.0,0.039732
1,All proteins + Age,continuous_SIS3,RidgeCV,yes_inside_LOOCV,Age,85,1012,17.141044,13.736939,0.311375,0.003723,2.429084,0.226474,0.037139,1.430174,1000.0,464.158883,1000.0,0.063147



Running continuous scaled Ridge model: All proteins + Time Since Stroke
Clinical features: ['Time Since Stroke']
  All proteins + Time Since Stroke: finished 10/85 patients, elapsed 0.01 min
  All proteins + Time Since Stroke: finished 20/85 patients, elapsed 0.01 min
  All proteins + Time Since Stroke: finished 30/85 patients, elapsed 0.01 min
  All proteins + Time Since Stroke: finished 40/85 patients, elapsed 0.02 min
  All proteins + Time Since Stroke: finished 50/85 patients, elapsed 0.02 min
  All proteins + Time Since Stroke: finished 60/85 patients, elapsed 0.03 min
  All proteins + Time Since Stroke: finished 70/85 patients, elapsed 0.03 min
  All proteins + Time Since Stroke: finished 80/85 patients, elapsed 0.04 min

Finished: All proteins + Time Since Stroke
Runtime: 0.04 minutes


,model,outcome_type,estimator,scaling,clinical_features,n_patients,n_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,median_selected_alpha,min_selected_alpha,max_selected_alpha,runtime_minutes
0,All proteins only,continuous_SIS3,RidgeCV,yes_inside_LOOCV,none,85,1011,17.141062,13.733057,0.311306,0.003731,2.428143,0.227991,0.035854,1.445462,1000.0,464.158883,1000.0,0.039732
1,All proteins + Age,continuous_SIS3,RidgeCV,yes_inside_LOOCV,Age,85,1012,17.141044,13.736939,0.311375,0.003723,2.429084,0.226474,0.037139,1.430174,1000.0,464.158883,1000.0,0.063147
2,All proteins + Time Since Stroke,continuous_SIS3,RidgeCV,yes_inside_LOOCV,Time Since Stroke,85,1012,17.273137,13.853874,0.296049,0.005942,2.226094,0.225593,0.037902,1.421338,1000.0,215.443469,1000.0,0.038875



Running continuous scaled Ridge model: All proteins + Age + Time Since Stroke
Clinical features: ['Age', 'Time Since Stroke']
  All proteins + Age + Time Since Stroke: finished 10/85 patients, elapsed 0.00 min
  All proteins + Age + Time Since Stroke: finished 20/85 patients, elapsed 0.01 min
  All proteins + Age + Time Since Stroke: finished 30/85 patients, elapsed 0.01 min
  All proteins + Age + Time Since Stroke: finished 40/85 patients, elapsed 0.02 min
  All proteins + Age + Time Since Stroke: finished 50/85 patients, elapsed 0.02 min
  All proteins + Age + Time Since Stroke: finished 60/85 patients, elapsed 0.03 min
  All proteins + Age + Time Since Stroke: finished 70/85 patients, elapsed 0.03 min
  All proteins + Age + Time Since Stroke: finished 80/85 patients, elapsed 0.04 min

Finished: All proteins + Age + Time Since Stroke
Runtime: 0.04 minutes


,model,outcome_type,estimator,scaling,clinical_features,n_patients,n_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,median_selected_alpha,min_selected_alpha,max_selected_alpha,runtime_minutes
0,All proteins only,continuous_SIS3,RidgeCV,yes_inside_LOOCV,none,85,1011,17.141062,13.733057,0.311306,0.003731,2.428143,0.227991,0.035854,1.445462,1000.0,464.158883,1000.0,0.039732
1,All proteins + Age,continuous_SIS3,RidgeCV,yes_inside_LOOCV,Age,85,1012,17.141044,13.736939,0.311375,0.003723,2.429084,0.226474,0.037139,1.430174,1000.0,464.158883,1000.0,0.063147
2,All proteins + Time Since Stroke,continuous_SIS3,RidgeCV,yes_inside_LOOCV,Time Since Stroke,85,1012,17.273137,13.853874,0.296049,0.005942,2.226094,0.225593,0.037902,1.421338,1000.0,215.443469,1000.0,0.038875
3,All proteins + Age + Time Since Stroke,continuous_SIS3,RidgeCV,yes_inside_LOOCV,"Age, Time Since Stroke",85,1013,17.224915,13.804587,0.301753,0.005007,2.300413,0.230145,0.034095,1.467311,1000.0,215.443469,1000.0,0.039813



All continuous scaled Ridge models finished
Total runtime: 0.19 minutes
Saved: strokecog_continuous_scaled_ridge_results_final.csv


,model,outcome_type,estimator,scaling,clinical_features,n_patients,n_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,median_selected_alpha,min_selected_alpha,max_selected_alpha,runtime_minutes
0,All proteins only,continuous_SIS3,RidgeCV,yes_inside_LOOCV,none,85,1011,17.141062,13.733057,0.311306,0.003731,2.428143,0.227991,0.035854,1.445462,1000.0,464.158883,1000.0,0.039732
1,All proteins + Age,continuous_SIS3,RidgeCV,yes_inside_LOOCV,Age,85,1012,17.141044,13.736939,0.311375,0.003723,2.429084,0.226474,0.037139,1.430174,1000.0,464.158883,1000.0,0.063147
2,All proteins + Time Since Stroke,continuous_SIS3,RidgeCV,yes_inside_LOOCV,Time Since Stroke,85,1012,17.273137,13.853874,0.296049,0.005942,2.226094,0.225593,0.037902,1.421338,1000.0,215.443469,1000.0,0.038875
3,All proteins + Age + Time Since Stroke,continuous_SIS3,RidgeCV,yes_inside_LOOCV,"Age, Time Since Stroke",85,1013,17.224915,13.804587,0.301753,0.005007,2.300413,0.230145,0.034095,1.467311,1000.0,215.443469,1000.0,0.039813


In [8]:
# ============================================================
# StrokeCog: Clinical-only baseline models
# Outcome: SIS3 as continuous mood score
# Predictors:
# 1. Mean-only baseline
# 2. Age only
# 3. Time Since Stroke only
# 4. Age + Time Since Stroke
# Scaling: inside LOOCV
# ============================================================

import time
import math
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from math import log10

# ------------------------------------------------------------
# 0. Load clinical data
# ------------------------------------------------------------
clinical = pd.read_csv("Subject_Clinical_Data.csv")

print("Clinical shape:", clinical.shape)
print(clinical[["PID", "SIS3", "Age", "Time Since Stroke"]].head())

assert "PID" in clinical.columns
assert "SIS3" in clinical.columns
assert "Age" in clinical.columns
assert "Time Since Stroke" in clinical.columns

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------
def safe_neg_log10(p):
    if p <= 0:
        return np.inf
    return -log10(p)


def calculate_metrics(true_vals, pred_vals, model_name, n_features):
    true_vals = np.array(true_vals, dtype=float)
    pred_vals = np.array(pred_vals, dtype=float)

    rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))
    mae = mean_absolute_error(true_vals, pred_vals)

    pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
    spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

    return {
        "model": model_name,
        "outcome_type": "continuous_SIS3",
        "n_patients": len(true_vals),
        "n_features": n_features,
        "RMSE": rmse,
        "MAE": mae,
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "-log10_Pearson_p": safe_neg_log10(pearson_p),
        "Spearman_r": spearman_r,
        "Spearman_p": spearman_p,
        "-log10_Spearman_p": safe_neg_log10(spearman_p)
    }


def run_mean_only_loocv():
    data = clinical[["PID", "SIS3"]].dropna().copy()

    true_vals = []
    pred_vals = []
    pid_vals = []

    for p in data["PID"].values:
        train = data[data["PID"] != p]
        test = data[data["PID"] == p]

        y_train = train["SIS3"].astype(float).values
        y_test = test["SIS3"].astype(float).values

        y_pred = np.mean(y_train)

        true_vals.append(float(y_test[0]))
        pred_vals.append(float(y_pred))
        pid_vals.append(p)

    result = calculate_metrics(
        true_vals=true_vals,
        pred_vals=pred_vals,
        model_name="Mean-only baseline",
        n_features=0
    )

    pred_table = pd.DataFrame({
        "PID": pid_vals,
        "SIS3_actual": true_vals,
        "SIS3_predicted": pred_vals
    })

    return result, pred_table


def run_clinical_only_loocv(clinical_features, model_name):
    needed_cols = ["PID", "SIS3"] + clinical_features
    data = clinical[needed_cols].dropna().copy()

    true_vals = []
    pred_vals = []
    pid_vals = []

    start = time.time()

    for i, p in enumerate(data["PID"].values, start=1):
        train = data[data["PID"] != p]
        test = data[data["PID"] == p]

        X_train = train[clinical_features].values
        y_train = train["SIS3"].astype(float).values

        X_test = test[clinical_features].values
        y_test = test["SIS3"].astype(float).values

        model = make_pipeline(
            StandardScaler(),
            LinearRegression()
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        true_vals.append(float(y_test[0]))
        pred_vals.append(float(y_pred[0]))
        pid_vals.append(p)

    result = calculate_metrics(
        true_vals=true_vals,
        pred_vals=pred_vals,
        model_name=model_name,
        n_features=len(clinical_features)
    )

    result["clinical_features"] = ", ".join(clinical_features)
    result["estimator"] = "LinearRegression"
    result["scaling"] = "yes_inside_LOOCV"
    result["runtime_seconds"] = time.time() - start

    pred_table = pd.DataFrame({
        "PID": pid_vals,
        "SIS3_actual": true_vals,
        "SIS3_predicted": pred_vals
    })

    return result, pred_table


# ------------------------------------------------------------
# 2. Run clinical-only baseline models
# ------------------------------------------------------------
results = []
prediction_tables = {}

# Mean-only baseline
result, pred_table = run_mean_only_loocv()
results.append(result)
prediction_tables["Mean-only baseline"] = pred_table

# Clinical-only models
model_specs = [
    (["Age"], "Age only"),
    (["Time Since Stroke"], "Time Since Stroke only"),
    (["Age", "Time Since Stroke"], "Age + Time Since Stroke")
]

for clinical_features, model_name in model_specs:
    print("Running:", model_name)
    result, pred_table = run_clinical_only_loocv(clinical_features, model_name)
    results.append(result)
    prediction_tables[model_name] = pred_table

    safe_name = model_name.replace(" ", "_").replace("+", "plus")
    pred_table.to_csv(f"strokecog_predictions_{safe_name}_clinical_only.csv", index=False)

# ------------------------------------------------------------
# 3. Save and display results
# ------------------------------------------------------------
clinical_only_results = pd.DataFrame(results)
clinical_only_results.to_csv("strokecog_clinical_only_baseline_results.csv", index=False)

display(clinical_only_results)

Clinical shape: (85, 14)
   PID  SIS3  Age  Time Since Stroke
0    1    89   66                228
1    3    67   80               2862
2    4    28   49               2111
3    6    92   49               1104
4    8    75   74               1238
Running: Age only
Running: Time Since Stroke only
Running: Age + Time Since Stroke


,model,outcome_type,n_patients,n_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,clinical_features,estimator,scaling,runtime_seconds
0,Mean-only baseline,continuous_SIS3,85,0,18.207243,14.331933,-1.000000,0.000000,inf,-1.000000,0.000000,inf,NaN,NaN,NaN,NaN
1,Age only,continuous_SIS3,85,1,18.279126,14.517936,-0.020515,0.852166,0.069476,0.042848,0.697006,0.156763,Age,LinearRegression,yes_inside_LOOCV,0.306387
2,Time Since Stroke only,continuous_SIS3,85,1,17.704913,14.305282,0.209987,0.053745,1.269662,0.087434,0.426213,0.370374,Time Since Stroke,LinearRegression,yes_inside_LOOCV,0.317418
3,Age + Time Since Stroke,continuous_SIS3,85,2,17.771052,14.415921,0.206989,0.057337,1.241563,0.180379,0.098541,1.006381,"Age, Time Since Stroke",LinearRegression,yes_inside_LOOCV,0.292397


In [9]:
# ============================================================
# StrokeCog: Univariate Spearman analysis
# Each protein vs SIS3 mood score
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Load data
clinical = pd.read_csv("Subject_Clinical_Data.csv")
npx = pd.read_csv("C_NPX_data.csv")

# Merge SIS3 with NPX data
data = clinical[["PID", "SIS3"]].merge(npx, on="PID", how="inner")

# Use all protein columns, including those with missing values
protein_cols_all = [c for c in npx.columns if c != "PID"]

print("Number of proteins tested:", len(protein_cols_all))

rows = []

for protein in protein_cols_all:
    x = data[protein]
    y = data["SIS3"]

    rho, pval = spearmanr(x, y, nan_policy="omit")

    rows.append({
        "protein": protein,
        "spearman_rho": rho,
        "p_value": pval,
        "direction": "higher_with_worse_mood" if rho < 0 else "higher_with_better_mood"
    })

spearman_results = pd.DataFrame(rows)

# Remove failed rows if any
spearman_results = spearman_results.dropna(subset=["p_value"]).copy()

# FDR correction
spearman_results["FDR_q"] = multipletests(
    spearman_results["p_value"],
    method="fdr_bh"
)[1]

# Sort
spearman_results = spearman_results.sort_values("p_value").reset_index(drop=True)

# Summary
n_total = len(spearman_results)
n_p05 = (spearman_results["p_value"] < 0.05).sum()
n_fdr05 = (spearman_results["FDR_q"] < 0.05).sum()
n_higher_worse = ((spearman_results["p_value"] < 0.05) &
                  (spearman_results["spearman_rho"] < 0)).sum()
n_higher_better = ((spearman_results["p_value"] < 0.05) &
                   (spearman_results["spearman_rho"] > 0)).sum()

print("Total proteins tested:", n_total)
print("Proteins with unadjusted p < 0.05:", n_p05)
print("Proteins with FDR q < 0.05:", n_fdr05)
print("Nominal proteins higher with worse mood:", n_higher_worse)
print("Nominal proteins higher with better mood:", n_higher_better)

display(spearman_results.head(20))

# Save
spearman_results.to_csv("strokecog_spearman_protein_sis3_results.csv", index=False)

Number of proteins tested: 1196
Total proteins tested: 1196
Proteins with unadjusted p < 0.05: 201
Proteins with FDR q < 0.05: 0
Nominal proteins higher with worse mood: 166
Nominal proteins higher with better mood: 35


,protein,spearman_rho,p_value,direction,FDR_q
0,OID01342,-0.386747,0.000256,higher_with_worse_mood,0.180378
1,OID05451,-0.377388,0.000370,higher_with_worse_mood,0.180378
2,OID00671,0.367580,0.000539,higher_with_better_mood,0.180378
3,OID01021,0.359915,0.000716,higher_with_better_mood,0.180378
4,OID01258,0.360556,0.000754,higher_with_better_mood,0.180378
5,OID01105,0.352613,0.000934,higher_with_better_mood,0.186112
6,OID00954,-0.340034,0.001453,higher_with_worse_mood,0.202687
7,OID01113,-0.339046,0.001503,higher_with_worse_mood,0.202687
8,OID05208,-0.333074,0.001841,higher_with_worse_mood,0.202687
9,OID00391,-0.328474,0.002146,higher_with_worse_mood,0.202687


In [10]:
# ============================================================
# StrokeCog: Reduced protein-set prediction
# Use nominal Spearman proteins only: p < 0.05
# Outcome: continuous SIS3
# Model: scaled Ridge regression inside LOOCV
# ============================================================

import re
import time
import math
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from math import log10

# ------------------------------------------------------------
# 0. Load files
# ------------------------------------------------------------
clinical = pd.read_csv("Subject_Clinical_Data.csv")
npx = pd.read_csv("C_NPX_data.csv")
spearman_results = pd.read_csv("strokecog_spearman_protein_sis3_results.csv")

# ------------------------------------------------------------
# 1. Select nominally associated proteins
# ------------------------------------------------------------
nominal_proteins = spearman_results.loc[
    spearman_results["p_value"] < 0.05, "protein"
].tolist()

print("Nominal p < 0.05 proteins:", len(nominal_proteins))

# For ML, remove protein columns with any missing values
NPX_clean = npx.dropna(axis=1).copy()

available_proteins = [p for p in nominal_proteins if p in NPX_clean.columns]

print("Nominal proteins available for ML after dropping missing:", len(available_proteins))

# ------------------------------------------------------------
# 2. Helper function
# ------------------------------------------------------------
def safe_neg_log10(p):
    if p <= 0:
        return np.inf
    return -log10(p)


def run_reduced_protein_ridge_loocv(protein_features, model_name):
    data = clinical[["PID", "SIS3"]].dropna().merge(
        NPX_clean[["PID"] + protein_features],
        on="PID",
        how="inner"
    )

    true_vals = []
    pred_vals = []
    pid_vals = []
    alpha_vals = []

    alphas = np.logspace(-3, 5, 25)

    start = time.time()

    for i, p in enumerate(data["PID"].values, start=1):
        train = data[data["PID"] != p]
        test = data[data["PID"] == p]

        X_train = train[protein_features].values
        y_train = train["SIS3"].astype(float).values

        X_test = test[protein_features].values
        y_test = test["SIS3"].astype(float).values

        model = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=alphas)
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        true_vals.append(float(y_test[0]))
        pred_vals.append(float(y_pred[0]))
        pid_vals.append(p)

        alpha_vals.append(model.named_steps["ridgecv"].alpha_)

        if i % 10 == 0:
            elapsed = (time.time() - start) / 60
            print(f"Finished {i}/{len(data)} patients, elapsed {elapsed:.2f} min")

    true_vals = np.array(true_vals)
    pred_vals = np.array(pred_vals)

    rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))
    mae = mean_absolute_error(true_vals, pred_vals)
    pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
    spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

    result = pd.DataFrame([{
        "model": model_name,
        "outcome_type": "continuous_SIS3",
        "estimator": "RidgeCV",
        "scaling": "yes_inside_LOOCV",
        "n_patients": len(true_vals),
        "n_features": len(protein_features),
        "RMSE": rmse,
        "MAE": mae,
        "Pearson_r": pearson_r,
        "Pearson_p": pearson_p,
        "-log10_Pearson_p": safe_neg_log10(pearson_p),
        "Spearman_r": spearman_r,
        "Spearman_p": spearman_p,
        "-log10_Spearman_p": safe_neg_log10(spearman_p),
        "median_selected_alpha": np.median(alpha_vals),
        "min_selected_alpha": np.min(alpha_vals),
        "max_selected_alpha": np.max(alpha_vals)
    }])

    pred_table = pd.DataFrame({
        "PID": pid_vals,
        "SIS3_actual": true_vals,
        "SIS3_predicted": pred_vals,
        "selected_alpha": alpha_vals
    })

    return result, pred_table


# ------------------------------------------------------------
# 3. Run reduced protein model
# ------------------------------------------------------------
result_reduced, pred_reduced = run_reduced_protein_ridge_loocv(
    protein_features=available_proteins,
    model_name="Nominal Spearman proteins only"
)

display(result_reduced)

result_reduced.to_csv("strokecog_nominal_spearman_proteins_ridge_result.csv", index=False)
pred_reduced.to_csv("strokecog_nominal_spearman_proteins_ridge_predictions.csv", index=False)

Nominal p < 0.05 proteins: 201
Nominal proteins available for ML after dropping missing: 180
Finished 10/85 patients, elapsed 0.00 min
Finished 20/85 patients, elapsed 0.01 min
Finished 30/85 patients, elapsed 0.01 min
Finished 40/85 patients, elapsed 0.02 min
Finished 50/85 patients, elapsed 0.03 min
Finished 60/85 patients, elapsed 0.04 min
Finished 70/85 patients, elapsed 0.04 min
Finished 80/85 patients, elapsed 0.05 min


,model,outcome_type,estimator,scaling,n_patients,n_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,median_selected_alpha,min_selected_alpha,max_selected_alpha
0,Nominal Spearman proteins only,continuous_SIS3,RidgeCV,yes_inside_LOOCV,85,180,14.904792,11.648056,0.560316,2.459058e-08,7.609231,0.508901,6.588861e-07,6.18119,215.443469,100.0,215.443469


In [11]:
# ============================================================
# StrokeCog: Leakage-free reduced-protein prediction
# Feature selection happens INSIDE each LOOCV fold
# Outcome: continuous SIS3
# Model: scaled Ridge regression
# ============================================================

import time
import math
import numpy as np
import pandas as pd

from scipy.stats import spearmanr, pearsonr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import log10

# ------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------
clinical = pd.read_csv("Subject_Clinical_Data.csv")
npx = pd.read_csv("C_NPX_data.csv")

data = clinical[["PID", "SIS3"]].merge(npx, on="PID", how="inner")

protein_cols_all = [c for c in npx.columns if c != "PID"]

print("Patients:", data.shape[0])
print("Proteins available before missing filtering:", len(protein_cols_all))

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------
def safe_neg_log10(p):
    if p <= 0:
        return np.inf
    return -log10(p)


def select_spearman_proteins_training_only(train_data, protein_cols, p_threshold=0.05):
    """
    Select proteins using only the training patients.
    Proteins with missing values in the training set are skipped for prediction simplicity.
    """
    selected = []

    for protein in protein_cols:
        x = train_data[protein]
        y = train_data["SIS3"]

        # Skip proteins with missing values in training data
        if x.isna().any():
            continue

        rho, pval = spearmanr(x, y, nan_policy="omit")

        if not np.isnan(pval) and pval < p_threshold:
            selected.append(protein)

    return selected


# ------------------------------------------------------------
# 2. Leakage-free LOOCV
# ------------------------------------------------------------
true_vals = []
pred_vals = []
pid_vals = []
n_selected_each_fold = []
selected_proteins_each_fold = []
alpha_vals = []

alphas = np.logspace(-3, 5, 25)

start = time.time()

for i, heldout_pid in enumerate(data["PID"].values, start=1):

    train = data[data["PID"] != heldout_pid].copy()
    test = data[data["PID"] == heldout_pid].copy()

    # Select proteins using training data only
    selected_proteins = select_spearman_proteins_training_only(
        train_data=train,
        protein_cols=protein_cols_all,
        p_threshold=0.05
    )

    n_selected_each_fold.append(len(selected_proteins))
    selected_proteins_each_fold.append(selected_proteins)

    y_train = train["SIS3"].astype(float).values
    y_test = test["SIS3"].astype(float).values

    # If no proteins selected, predict training mean
    if len(selected_proteins) == 0:
        y_pred = np.array([np.mean(y_train)])
        alpha_vals.append(np.nan)

    else:
        X_train = train[selected_proteins].values
        X_test = test[selected_proteins].values

        model = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=alphas)
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        alpha_vals.append(model.named_steps["ridgecv"].alpha_)

    true_vals.append(float(y_test[0]))
    pred_vals.append(float(y_pred[0]))
    pid_vals.append(heldout_pid)

    if i % 10 == 0:
        elapsed = (time.time() - start) / 60
        print(
            f"Finished {i}/{len(data)} patients, "
            f"selected proteins this fold = {len(selected_proteins)}, "
            f"elapsed {elapsed:.2f} min"
        )

# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------
true_vals = np.array(true_vals, dtype=float)
pred_vals = np.array(pred_vals, dtype=float)

rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))
mae = mean_absolute_error(true_vals, pred_vals)

pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

leakage_free_result = pd.DataFrame([{
    "model": "Leakage-free nominal Spearman proteins inside LOOCV",
    "outcome_type": "continuous_SIS3",
    "estimator": "RidgeCV",
    "scaling": "yes_inside_LOOCV",
    "feature_selection": "Spearman_p_lt_0.05_inside_each_training_fold",
    "n_patients": len(true_vals),
    "mean_selected_features": np.mean(n_selected_each_fold),
    "median_selected_features": np.median(n_selected_each_fold),
    "min_selected_features": np.min(n_selected_each_fold),
    "max_selected_features": np.max(n_selected_each_fold),
    "RMSE": rmse,
    "MAE": mae,
    "Pearson_r": pearson_r,
    "Pearson_p": pearson_p,
    "-log10_Pearson_p": safe_neg_log10(pearson_p),
    "Spearman_r": spearman_r,
    "Spearman_p": spearman_p,
    "-log10_Spearman_p": safe_neg_log10(spearman_p),
    "median_selected_alpha": np.nanmedian(alpha_vals)
}])

pred_table = pd.DataFrame({
    "PID": pid_vals,
    "SIS3_actual": true_vals,
    "SIS3_predicted": pred_vals,
    "n_selected_features": n_selected_each_fold,
    "selected_alpha": alpha_vals
})

# ------------------------------------------------------------
# 4. Protein selection stability
# ------------------------------------------------------------
all_selected = []
for fold_list in selected_proteins_each_fold:
    all_selected.extend(fold_list)

selection_counts = pd.Series(all_selected).value_counts().reset_index()
selection_counts.columns = ["protein", "selected_in_n_folds"]

selection_counts["selection_frequency"] = (
    selection_counts["selected_in_n_folds"] / len(data)
)

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------
leakage_free_result.to_csv(
    "strokecog_leakage_free_spearman_ridge_result.csv",
    index=False
)

pred_table.to_csv(
    "strokecog_leakage_free_spearman_ridge_predictions.csv",
    index=False
)

selection_counts.to_csv(
    "strokecog_leakage_free_spearman_selection_stability.csv",
    index=False
)

display(leakage_free_result)
display(selection_counts.head(20))

Patients: 85
Proteins available before missing filtering: 1196
Finished 10/85 patients, selected proteins this fold = 181, elapsed 0.14 min


ValueError: Input X contains NaN.
RidgeCV does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [12]:
# ============================================================
# StrokeCog: Leakage-free reduced-protein prediction
# Feature selection happens INSIDE each LOOCV fold
# Missing values handled by SimpleImputer inside each fold
# Outcome: continuous SIS3
# Model: SimpleImputer + StandardScaler + RidgeCV
# ============================================================

import time
import math
import warnings
import numpy as np
import pandas as pd

from scipy.stats import spearmanr, pearsonr, ConstantInputWarning
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import log10

# ------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------
clinical = pd.read_csv("Subject_Clinical_Data.csv")
npx = pd.read_csv("C_NPX_data.csv")

data = clinical[["PID", "SIS3"]].merge(npx, on="PID", how="inner")

protein_cols_all = [c for c in npx.columns if c != "PID"]

print("Patients:", data.shape[0])
print("Proteins available before missing filtering:", len(protein_cols_all))

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------
def safe_neg_log10(p):
    if p <= 0:
        return np.inf
    return -log10(p)


def select_spearman_proteins_training_only(
    train_data,
    protein_cols,
    p_threshold=0.05,
    min_nonmissing=20
):
    """
    Select proteins using only the training patients.
    Missing values are allowed for Spearman calculation using nan_policy='omit'.
    Later, missing values are handled by SimpleImputer inside the ML pipeline.
    """
    selected = []

    y = train_data["SIS3"].astype(float)

    for protein in protein_cols:
        x = train_data[protein]

        # Require enough non-missing values
        valid = x.notna() & y.notna()
        if valid.sum() < min_nonmissing:
            continue

        # Skip constant proteins in the training fold
        if x[valid].nunique() <= 1:
            continue

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConstantInputWarning)
            rho, pval = spearmanr(x, y, nan_policy="omit")

        if not np.isnan(pval) and pval < p_threshold:
            selected.append(protein)

    return selected


# ------------------------------------------------------------
# 2. Leakage-free LOOCV
# ------------------------------------------------------------
true_vals = []
pred_vals = []
pid_vals = []
n_selected_each_fold = []
selected_proteins_each_fold = []
alpha_vals = []

alphas = np.logspace(-3, 5, 25)

start = time.time()

for i, heldout_pid in enumerate(data["PID"].values, start=1):

    train = data[data["PID"] != heldout_pid].copy()
    test = data[data["PID"] == heldout_pid].copy()

    # Select proteins using training data only
    selected_proteins = select_spearman_proteins_training_only(
        train_data=train,
        protein_cols=protein_cols_all,
        p_threshold=0.05,
        min_nonmissing=20
    )

    n_selected_each_fold.append(len(selected_proteins))
    selected_proteins_each_fold.append(selected_proteins)

    y_train = train["SIS3"].astype(float).values
    y_test = test["SIS3"].astype(float).values

    # If no proteins selected, predict training mean
    if len(selected_proteins) == 0:
        y_pred = np.array([np.mean(y_train)])
        alpha_vals.append(np.nan)

    else:
        X_train = train[selected_proteins].values
        X_test = test[selected_proteins].values

        model = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            RidgeCV(alphas=alphas)
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        alpha_vals.append(model.named_steps["ridgecv"].alpha_)

    true_vals.append(float(y_test[0]))
    pred_vals.append(float(y_pred[0]))
    pid_vals.append(heldout_pid)

    if i % 10 == 0:
        elapsed = (time.time() - start) / 60
        print(
            f"Finished {i}/{len(data)} patients, "
            f"selected proteins this fold = {len(selected_proteins)}, "
            f"elapsed {elapsed:.2f} min"
        )

# ------------------------------------------------------------
# 3. Metrics
# ------------------------------------------------------------
true_vals = np.array(true_vals, dtype=float)
pred_vals = np.array(pred_vals, dtype=float)

rmse = math.sqrt(mean_squared_error(true_vals, pred_vals))
mae = mean_absolute_error(true_vals, pred_vals)

pearson_r, pearson_p = pearsonr(pred_vals, true_vals)
spearman_r, spearman_p = spearmanr(pred_vals, true_vals)

leakage_free_result = pd.DataFrame([{
    "model": "Leakage-free nominal Spearman proteins inside LOOCV",
    "outcome_type": "continuous_SIS3",
    "estimator": "RidgeCV",
    "imputation": "median_inside_LOOCV",
    "scaling": "yes_inside_LOOCV",
    "feature_selection": "Spearman_p_lt_0.05_inside_each_training_fold",
    "n_patients": len(true_vals),
    "mean_selected_features": np.mean(n_selected_each_fold),
    "median_selected_features": np.median(n_selected_each_fold),
    "min_selected_features": np.min(n_selected_each_fold),
    "max_selected_features": np.max(n_selected_each_fold),
    "RMSE": rmse,
    "MAE": mae,
    "Pearson_r": pearson_r,
    "Pearson_p": pearson_p,
    "-log10_Pearson_p": safe_neg_log10(pearson_p),
    "Spearman_r": spearman_r,
    "Spearman_p": spearman_p,
    "-log10_Spearman_p": safe_neg_log10(spearman_p),
    "median_selected_alpha": np.nanmedian(alpha_vals)
}])

pred_table = pd.DataFrame({
    "PID": pid_vals,
    "SIS3_actual": true_vals,
    "SIS3_predicted": pred_vals,
    "n_selected_features": n_selected_each_fold,
    "selected_alpha": alpha_vals
})

# ------------------------------------------------------------
# 4. Protein selection stability
# ------------------------------------------------------------
all_selected = []
for fold_list in selected_proteins_each_fold:
    all_selected.extend(fold_list)

selection_counts = pd.Series(all_selected).value_counts().reset_index()
selection_counts.columns = ["protein", "selected_in_n_folds"]

selection_counts["selection_frequency"] = (
    selection_counts["selected_in_n_folds"] / len(data)
)

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------
leakage_free_result.to_csv(
    "strokecog_leakage_free_spearman_ridge_result.csv",
    index=False
)

pred_table.to_csv(
    "strokecog_leakage_free_spearman_ridge_predictions.csv",
    index=False
)

selection_counts.to_csv(
    "strokecog_leakage_free_spearman_selection_stability.csv",
    index=False
)

display(leakage_free_result)
display(selection_counts.head(20))

Patients: 85
Proteins available before missing filtering: 1196
Finished 10/85 patients, selected proteins this fold = 203, elapsed 0.30 min
Finished 20/85 patients, selected proteins this fold = 214, elapsed 0.60 min
Finished 30/85 patients, selected proteins this fold = 198, elapsed 0.89 min
Finished 40/85 patients, selected proteins this fold = 183, elapsed 1.18 min
Finished 50/85 patients, selected proteins this fold = 208, elapsed 1.49 min
Finished 60/85 patients, selected proteins this fold = 189, elapsed 1.77 min
Finished 70/85 patients, selected proteins this fold = 202, elapsed 2.07 min
Finished 80/85 patients, selected proteins this fold = 200, elapsed 2.35 min


,model,outcome_type,estimator,imputation,scaling,feature_selection,n_patients,mean_selected_features,median_selected_features,min_selected_features,max_selected_features,RMSE,MAE,Pearson_r,Pearson_p,-log10_Pearson_p,Spearman_r,Spearman_p,-log10_Spearman_p,median_selected_alpha
0,Leakage-free nominal Spearman proteins inside ...,continuous_SIS3,RidgeCV,median_inside_LOOCV,yes_inside_LOOCV,Spearman_p_lt_0.05_inside_each_training_fold,85,197.870588,201.0,164,214,18.249128,14.361168,0.203446,0.061833,1.208778,0.132639,0.226234,0.645441,215.443469


,protein,selected_in_n_folds,selection_frequency
0,OID01113,85,1.0
1,OID00608,85,1.0
2,OID00605,85,1.0
3,OID00592,85,1.0
4,OID00591,85,1.0
5,OID00467,85,1.0
6,OID00424,85,1.0
7,OID00419,85,1.0
8,OID00401,85,1.0
9,OID00392,85,1.0


In [13]:
selection_counts = pd.read_csv("strokecog_leakage_free_spearman_selection_stability.csv")
spearman_results = pd.read_csv("strokecog_spearman_protein_sis3_results.csv")

stable_table = selection_counts.merge(
    spearman_results,
    on="protein",
    how="left"
)

stable_table = stable_table.sort_values(
    ["selected_in_n_folds", "p_value"],
    ascending=[False, True]
)

display(stable_table.head(30))

stable_table.to_csv("strokecog_stable_selected_proteins_with_spearman.csv", index=False)

,protein,selected_in_n_folds,selection_frequency,spearman_rho,p_value,direction,FDR_q
38,OID01342,85,1.0,-0.386747,0.000256,higher_with_worse_mood,0.180378
62,OID05451,85,1.0,-0.377388,0.000370,higher_with_worse_mood,0.180378
97,OID00671,85,1.0,0.367580,0.000539,higher_with_better_mood,0.180378
72,OID01021,85,1.0,0.359915,0.000716,higher_with_better_mood,0.180378
136,OID01258,85,1.0,0.360556,0.000754,higher_with_better_mood,0.180378
13,OID01105,85,1.0,0.352613,0.000934,higher_with_better_mood,0.186112
52,OID00954,85,1.0,-0.340034,0.001453,higher_with_worse_mood,0.202687
0,OID01113,85,1.0,-0.339046,0.001503,higher_with_worse_mood,0.202687
119,OID05208,85,1.0,-0.333074,0.001841,higher_with_worse_mood,0.202687
10,OID00391,85,1.0,-0.328474,0.002146,higher_with_worse_mood,0.202687


In [14]:
import os
import pandas as pd

for root, dirs, files in os.walk("."):
    for f in files:
        if f.endswith((".csv", ".xlsx", ".txt")):
            print(os.path.join(root, f))

./strokecog_continuous_scaled_ridge_results_partial.csv
./strokecog_nominal_spearman_proteins_ridge_result.csv
./strokecog_predictions_All_proteins_only_scaled_lbfgs_final.csv
./strokecog_predictions_All_proteins_Age_Time_Since_Stroke_continuous_scaled_ridge.csv
./strokecog_predictions_All_proteins_Time_Since_Stroke_scaled_lbfgs_final.csv
./strokecog_scaled_lbfgs_results_partial.csv
./strokecog_predictions_All_proteins_Age_Time_Since_Stroke_fast_unscaled_checkpoint.csv
./Subject_Clinical_Data.csv
./strokecog_predictions_All_proteins_Age_Time_Since_Stroke_scaled_lbfgs_final.csv
./strokecog_predictions_Time_Since_Stroke_only_clinical_only.csv
./strokecog_predictions_Age_only_clinical_only.csv
./strokecog_continuous_scaled_ridge_results_final.csv
./C_NPX_data.csv
./strokecog_remaining_model_fast_unscaled_result.csv
./strokecog_spearman_protein_sis3_results.csv
./strokecog_clinical_only_baseline_results.csv
./strokecog_leakage_free_spearman_ridge_predictions.csv
./strokecog_leakage_free_sp

In [15]:
excel_file = "20191510_Buckwalter_NPX.xlsx"

xls = pd.ExcelFile(excel_file)
print(xls.sheet_names)

FileNotFoundError: [Errno 2] No such file or directory: '20191510_Buckwalter_NPX.xlsx'

In [17]:
import pandas as pd

sis3_corr = pd.read_csv("sis3_highly_corr_prs.csv")

print(sis3_corr.shape)
print(sis3_corr.columns.tolist())
display(sis3_corr.head(20))

(180, 3)
['panel', 'name', 'OID']


,panel,name,OID
0,Olink CARDIOVASCULAR II(v.5006),ANGPT1,OID00380
1,Olink CARDIOVASCULAR II(v.5006),CD40-L,OID00382
2,Olink CARDIOVASCULAR II(v.5006),SRC,OID00388
3,Olink CARDIOVASCULAR II(v.5006),IL-1ra,OID00389
4,Olink CARDIOVASCULAR II(v.5006),TNFRSF10A,OID00391
5,Olink CARDIOVASCULAR II(v.5006),STK4,OID00392
6,Olink CARDIOVASCULAR II(v.5006),PDGF subunit B,OID00401
7,Olink CARDIOVASCULAR II(v.5006),GH,OID00417
8,Olink CARDIOVASCULAR II(v.5006),GLO1,OID00419
9,Olink CARDIOVASCULAR II(v.5006),DECR1,OID00424


In [18]:
stable_table = pd.read_csv("strokecog_stable_selected_proteins_with_spearman.csv")

print("Stable table columns:")
print(stable_table.columns.tolist())

print("sis3_corr columns:")
print(sis3_corr.columns.tolist())

Stable table columns:
['protein', 'selected_in_n_folds', 'selection_frequency', 'spearman_rho', 'p_value', 'direction', 'FDR_q']
sis3_corr columns:
['panel', 'name', 'OID']


In [19]:
# Adjust column name if needed
sis3_corr_mapping = sis3_corr.rename(columns={
    "OID": "protein"
})

stable_annotated = stable_table.merge(
    sis3_corr_mapping,
    on="protein",
    how="left"
)

display(stable_annotated.head(30))

stable_annotated.to_csv(
    "strokecog_stable_selected_proteins_annotated.csv",
    index=False
)

,protein,selected_in_n_folds,selection_frequency,spearman_rho,p_value,direction,FDR_q,panel,name
0,OID01342,85,1.0,-0.386747,0.000256,higher_with_worse_mood,0.180378,Olink CELL REGULATION(v.3702),DNAJB1
1,OID05451,85,1.0,-0.377388,0.000370,higher_with_worse_mood,0.180378,Olink ONCOLOGY III(v.4001),TXNDC15
2,OID00671,85,1.0,0.367580,0.000539,higher_with_better_mood,0.180378,Olink ONCOLOGY II(v.7004),ITGAV
3,OID01021,85,1.0,0.359915,0.000716,higher_with_better_mood,0.180378,Olink IMMUNE RESPONSE(v.3203),ITGA11
4,OID01258,85,1.0,0.360556,0.000754,higher_with_better_mood,0.180378,NaN,NaN
5,OID01105,85,1.0,0.352613,0.000934,higher_with_better_mood,0.186112,Olink ORGAN DAMAGE(v.3311),HPGDS
6,OID00954,85,1.0,-0.340034,0.001453,higher_with_worse_mood,0.202687,Olink IMMUNE RESPONSE(v.3203),FGF2
7,OID01113,85,1.0,-0.339046,0.001503,higher_with_worse_mood,0.202687,Olink ORGAN DAMAGE(v.3311),PVALB
8,OID05208,85,1.0,-0.333074,0.001841,higher_with_worse_mood,0.202687,Olink NEURO EXPLORATORY(v.3911),RNF31
9,OID00391,85,1.0,-0.328474,0.002146,higher_with_worse_mood,0.202687,Olink CARDIOVASCULAR II(v.5006),TNFRSF10A


In [20]:
# ============================================================
# Step 1: Inspect SIS3 highly correlated protein file
# ============================================================

import pandas as pd

sis3_corr = pd.read_csv("sis3_highly_corr_prs.csv")

print("sis3_corr shape:", sis3_corr.shape)
print("Columns:")
print(sis3_corr.columns.tolist())

display(sis3_corr.head(20))

sis3_corr shape: (180, 3)
Columns:
['panel', 'name', 'OID']


,panel,name,OID
0,Olink CARDIOVASCULAR II(v.5006),ANGPT1,OID00380
1,Olink CARDIOVASCULAR II(v.5006),CD40-L,OID00382
2,Olink CARDIOVASCULAR II(v.5006),SRC,OID00388
3,Olink CARDIOVASCULAR II(v.5006),IL-1ra,OID00389
4,Olink CARDIOVASCULAR II(v.5006),TNFRSF10A,OID00391
5,Olink CARDIOVASCULAR II(v.5006),STK4,OID00392
6,Olink CARDIOVASCULAR II(v.5006),PDGF subunit B,OID00401
7,Olink CARDIOVASCULAR II(v.5006),GH,OID00417
8,Olink CARDIOVASCULAR II(v.5006),GLO1,OID00419
9,Olink CARDIOVASCULAR II(v.5006),DECR1,OID00424


In [21]:
# ============================================================
# Step 2: Load stable selected protein table
# ============================================================

stable_table = pd.read_csv("strokecog_stable_selected_proteins_with_spearman.csv")

print("stable_table shape:", stable_table.shape)
print("stable_table columns:")
print(stable_table.columns.tolist())

display(stable_table.head())

stable_table shape: (278, 7)
stable_table columns:
['protein', 'selected_in_n_folds', 'selection_frequency', 'spearman_rho', 'p_value', 'direction', 'FDR_q']


,protein,selected_in_n_folds,selection_frequency,spearman_rho,p_value,direction,FDR_q
0,OID01342,85,1.0,-0.386747,0.000256,higher_with_worse_mood,0.180378
1,OID05451,85,1.0,-0.377388,0.000370,higher_with_worse_mood,0.180378
2,OID00671,85,1.0,0.367580,0.000539,higher_with_better_mood,0.180378
3,OID01021,85,1.0,0.359915,0.000716,higher_with_better_mood,0.180378
4,OID01258,85,1.0,0.360556,0.000754,higher_with_better_mood,0.180378


In [22]:
# ============================================================
# Step 3: Find possible ID columns
# ============================================================

for col in sis3_corr.columns:
    print(col, "example values:", sis3_corr[col].dropna().astype(str).head(5).tolist())

panel example values: ['Olink CARDIOVASCULAR II(v.5006)', 'Olink CARDIOVASCULAR II(v.5006)', 'Olink CARDIOVASCULAR II(v.5006)', 'Olink CARDIOVASCULAR II(v.5006)', 'Olink CARDIOVASCULAR II(v.5006)']
name example values: ['ANGPT1', 'CD40-L', 'SRC', 'IL-1ra', 'TNFRSF10A']
OID example values: ['OID00380', 'OID00382', 'OID00388', 'OID00389', 'OID00391']


In [23]:
# ============================================================
# Step 4: Merge annotation with stable protein table
# ============================================================

id_col = "OID"   # change this if the actual column name is different

sis3_corr_mapping = sis3_corr.rename(columns={id_col: "protein"})

stable_annotated = stable_table.merge(
    sis3_corr_mapping,
    on="protein",
    how="left"
)

print("Annotated table shape:", stable_annotated.shape)
display(stable_annotated.head(30))

stable_annotated.to_csv(
    "strokecog_stable_selected_proteins_annotated.csv",
    index=False
)

Annotated table shape: (278, 9)


,protein,selected_in_n_folds,selection_frequency,spearman_rho,p_value,direction,FDR_q,panel,name
0,OID01342,85,1.0,-0.386747,0.000256,higher_with_worse_mood,0.180378,Olink CELL REGULATION(v.3702),DNAJB1
1,OID05451,85,1.0,-0.377388,0.000370,higher_with_worse_mood,0.180378,Olink ONCOLOGY III(v.4001),TXNDC15
2,OID00671,85,1.0,0.367580,0.000539,higher_with_better_mood,0.180378,Olink ONCOLOGY II(v.7004),ITGAV
3,OID01021,85,1.0,0.359915,0.000716,higher_with_better_mood,0.180378,Olink IMMUNE RESPONSE(v.3203),ITGA11
4,OID01258,85,1.0,0.360556,0.000754,higher_with_better_mood,0.180378,NaN,NaN
5,OID01105,85,1.0,0.352613,0.000934,higher_with_better_mood,0.186112,Olink ORGAN DAMAGE(v.3311),HPGDS
6,OID00954,85,1.0,-0.340034,0.001453,higher_with_worse_mood,0.202687,Olink IMMUNE RESPONSE(v.3203),FGF2
7,OID01113,85,1.0,-0.339046,0.001503,higher_with_worse_mood,0.202687,Olink ORGAN DAMAGE(v.3311),PVALB
8,OID05208,85,1.0,-0.333074,0.001841,higher_with_worse_mood,0.202687,Olink NEURO EXPLORATORY(v.3911),RNF31
9,OID00391,85,1.0,-0.328474,0.002146,higher_with_worse_mood,0.202687,Olink CARDIOVASCULAR II(v.5006),TNFRSF10A


In [24]:
# ============================================================
# Step 5: Check annotation success
# ============================================================

missing_annotation = stable_annotated[
    stable_annotated.isna().any(axis=1)
]

print("Rows with any missing value:", missing_annotation.shape[0])

# More useful: check missing annotation based on likely name columns
print(stable_annotated.columns.tolist())

Rows with any missing value: 98
['protein', 'selected_in_n_folds', 'selection_frequency', 'spearman_rho', 'p_value', 'direction', 'FDR_q', 'panel', 'name']


In [25]:
# Count annotated proteins by panel and direction
panel_summary = (
    stable_annotated
    .dropna(subset=["panel"])
    .groupby(["panel", "direction"])
    .size()
    .reset_index(name="count")
    .sort_values(["panel", "count"], ascending=[True, False])
)

display(panel_summary)

panel_summary.to_csv("strokecog_panel_direction_summary.csv", index=False)

,panel,direction,count
1,Olink CARDIOVASCULAR II(v.5006),higher_with_worse_mood,14
0,Olink CARDIOVASCULAR II(v.5006),higher_with_better_mood,1
2,Olink CARDIOVASCULAR III(v.6113),higher_with_better_mood,4
3,Olink CARDIOVASCULAR III(v.6113),higher_with_worse_mood,3
5,Olink CELL REGULATION(v.3702),higher_with_worse_mood,18
4,Olink CELL REGULATION(v.3702),higher_with_better_mood,2
7,Olink DEVELOPMENT(v.3511),higher_with_worse_mood,9
6,Olink DEVELOPMENT(v.3511),higher_with_better_mood,5
9,Olink IMMUNE RESPONSE(v.3203),higher_with_worse_mood,20
8,Olink IMMUNE RESPONSE(v.3203),higher_with_better_mood,3


In [26]:
top_named = (
    stable_annotated
    .dropna(subset=["name"])
    .sort_values("p_value")
    [["protein", "name", "panel", "spearman_rho", "p_value", "FDR_q", "direction"]]
)

display(top_named.head(50))

top_named.to_csv("strokecog_top_named_stable_proteins.csv", index=False)

,protein,name,panel,spearman_rho,p_value,FDR_q,direction
0,OID01342,DNAJB1,Olink CELL REGULATION(v.3702),-0.386747,0.000256,0.180378,higher_with_worse_mood
1,OID05451,TXNDC15,Olink ONCOLOGY III(v.4001),-0.377388,0.000370,0.180378,higher_with_worse_mood
2,OID00671,ITGAV,Olink ONCOLOGY II(v.7004),0.367580,0.000539,0.180378,higher_with_better_mood
3,OID01021,ITGA11,Olink IMMUNE RESPONSE(v.3203),0.359915,0.000716,0.180378,higher_with_better_mood
5,OID01105,HPGDS,Olink ORGAN DAMAGE(v.3311),0.352613,0.000934,0.186112,higher_with_better_mood
6,OID00954,FGF2,Olink IMMUNE RESPONSE(v.3203),-0.340034,0.001453,0.202687,higher_with_worse_mood
7,OID01113,PVALB,Olink ORGAN DAMAGE(v.3311),-0.339046,0.001503,0.202687,higher_with_worse_mood
8,OID05208,RNF31,Olink NEURO EXPLORATORY(v.3911),-0.333074,0.001841,0.202687,higher_with_worse_mood
9,OID00391,TNFRSF10A,Olink CARDIOVASCULAR II(v.5006),-0.328474,0.002146,0.202687,higher_with_worse_mood
11,OID05141,PTPN1,Olink NEURO EXPLORATORY(v.3911),-0.322238,0.002634,0.202687,higher_with_worse_mood


In [27]:
# ============================================================
# StrokeCog: Biological theme classification of SIS3 proteins
# ============================================================

import pandas as pd
import numpy as np

# Load annotated table
stable_annotated = pd.read_csv("strokecog_stable_selected_proteins_annotated.csv")

# Keep named proteins only
named = stable_annotated.dropna(subset=["name"]).copy()

print("Named proteins:", named.shape[0])
display(named.head())

# ------------------------------------------------------------
# 1. Define simple biological theme rules
# ------------------------------------------------------------

immune_keywords = [
    "IL", "TNF", "TNFR", "TRIM", "IRF", "LY", "CD", "CXCL", "CCL",
    "IFN", "HLA", "TREM", "CSF", "NFKB"
]

integrin_ecm_keywords = [
    "ITGA", "ITGB", "COL", "MMP", "TIMP", "SDC", "VCAM", "ICAM",
    "ANGPT", "FGF", "EGF"
]

stress_cell_reg_keywords = [
    "DNAJ", "TXN", "STX", "SRPK", "RNF", "PTPN", "RARRES", "NAMPT"
]

metabolism_keywords = [
    "HPGDS", "SDC4", "ABHD", "CDHR", "NAMPT", "QDPR", "KYNU"
]

neuro_keywords = [
    "PVALB", "GDNF", "NRP", "EFEMP", "PTPN", "RNF"
]


def assign_theme(row):
    name = str(row["name"]).upper()
    panel = str(row["panel"]).upper()

    themes = []

    if any(k in name for k in immune_keywords) or "IMMUNE" in panel:
        themes.append("immune/inflammatory")

    if any(k in name for k in integrin_ecm_keywords):
        themes.append("integrin/ECM/vascular-remodeling")

    if any(k in name for k in stress_cell_reg_keywords) or "CELL REGULATION" in panel:
        themes.append("cell-stress/cell-regulation")

    if any(k in name for k in metabolism_keywords) or "METABOLISM" in panel:
        themes.append("metabolism/prostaglandin")

    if any(k in name for k in neuro_keywords) or "NEURO" in panel:
        themes.append("neuro-related")

    if len(themes) == 0:
        themes.append("other/uncurated")

    return "; ".join(themes)


named["biological_theme"] = named.apply(assign_theme, axis=1)

# ------------------------------------------------------------
# 2. Top named candidate proteins
# ------------------------------------------------------------

top_named = named.sort_values("p_value")[
    ["protein", "name", "panel", "spearman_rho", "p_value", "FDR_q", "direction", "biological_theme"]
]

display(top_named.head(40))

top_named.to_csv("strokecog_top_named_proteins_with_themes.csv", index=False)

# ------------------------------------------------------------
# 3. Theme summary
# ------------------------------------------------------------

theme_rows = []

for _, row in named.iterrows():
    themes = row["biological_theme"].split("; ")
    for theme in themes:
        theme_rows.append({
            "protein": row["protein"],
            "name": row["name"],
            "direction": row["direction"],
            "theme": theme
        })

theme_long = pd.DataFrame(theme_rows)

theme_summary = (
    theme_long
    .groupby(["theme", "direction"])
    .size()
    .reset_index(name="count")
    .sort_values(["theme", "count"], ascending=[True, False])
)

display(theme_summary)

theme_summary.to_csv("strokecog_theme_direction_summary.csv", index=False)

# ------------------------------------------------------------
# 4. Direction summary
# ------------------------------------------------------------

direction_summary = (
    named
    .groupby("direction")
    .size()
    .reset_index(name="count")
)

display(direction_summary)
direction_summary.to_csv("strokecog_named_direction_summary.csv", index=False)

Named proteins: 180


,protein,selected_in_n_folds,selection_frequency,spearman_rho,p_value,direction,FDR_q,panel,name
0,OID01342,85,1.0,-0.386747,0.000256,higher_with_worse_mood,0.180378,Olink CELL REGULATION(v.3702),DNAJB1
1,OID05451,85,1.0,-0.377388,0.000370,higher_with_worse_mood,0.180378,Olink ONCOLOGY III(v.4001),TXNDC15
2,OID00671,85,1.0,0.367580,0.000539,higher_with_better_mood,0.180378,Olink ONCOLOGY II(v.7004),ITGAV
3,OID01021,85,1.0,0.359915,0.000716,higher_with_better_mood,0.180378,Olink IMMUNE RESPONSE(v.3203),ITGA11
5,OID01105,85,1.0,0.352613,0.000934,higher_with_better_mood,0.186112,Olink ORGAN DAMAGE(v.3311),HPGDS


,protein,name,panel,spearman_rho,p_value,FDR_q,direction,biological_theme
0,OID01342,DNAJB1,Olink CELL REGULATION(v.3702),-0.386747,0.000256,0.180378,higher_with_worse_mood,cell-stress/cell-regulation
1,OID05451,TXNDC15,Olink ONCOLOGY III(v.4001),-0.377388,0.000370,0.180378,higher_with_worse_mood,cell-stress/cell-regulation
2,OID00671,ITGAV,Olink ONCOLOGY II(v.7004),0.367580,0.000539,0.180378,higher_with_better_mood,integrin/ECM/vascular-remodeling
3,OID01021,ITGA11,Olink IMMUNE RESPONSE(v.3203),0.359915,0.000716,0.180378,higher_with_better_mood,immune/inflammatory; integrin/ECM/vascular-rem...
5,OID01105,HPGDS,Olink ORGAN DAMAGE(v.3311),0.352613,0.000934,0.186112,higher_with_better_mood,metabolism/prostaglandin
6,OID00954,FGF2,Olink IMMUNE RESPONSE(v.3203),-0.340034,0.001453,0.202687,higher_with_worse_mood,immune/inflammatory; integrin/ECM/vascular-rem...
7,OID01113,PVALB,Olink ORGAN DAMAGE(v.3311),-0.339046,0.001503,0.202687,higher_with_worse_mood,neuro-related
8,OID05208,RNF31,Olink NEURO EXPLORATORY(v.3911),-0.333074,0.001841,0.202687,higher_with_worse_mood,cell-stress/cell-regulation; neuro-related
9,OID00391,TNFRSF10A,Olink CARDIOVASCULAR II(v.5006),-0.328474,0.002146,0.202687,higher_with_worse_mood,immune/inflammatory
11,OID05141,PTPN1,Olink NEURO EXPLORATORY(v.3911),-0.322238,0.002634,0.202687,higher_with_worse_mood,cell-stress/cell-regulation; neuro-related


,theme,direction,count
1,cell-stress/cell-regulation,higher_with_worse_mood,27
0,cell-stress/cell-regulation,higher_with_better_mood,2
3,immune/inflammatory,higher_with_worse_mood,32
2,immune/inflammatory,higher_with_better_mood,6
5,integrin/ECM/vascular-remodeling,higher_with_worse_mood,6
4,integrin/ECM/vascular-remodeling,higher_with_better_mood,5
7,metabolism/prostaglandin,higher_with_worse_mood,16
6,metabolism/prostaglandin,higher_with_better_mood,3
9,neuro-related,higher_with_worse_mood,27
8,neuro-related,higher_with_better_mood,1


,direction,count
0,higher_with_better_mood,25
1,higher_with_worse_mood,155


In [28]:
import pandas as pd

tables = []

# Clinical-only baseline
clinical_only = pd.read_csv("strokecog_clinical_only_baseline_results.csv")
clinical_only["analysis_group"] = "clinical_only"
tables.append(clinical_only)

# All-protein continuous Ridge
all_protein = pd.read_csv("strokecog_continuous_scaled_ridge_results_final.csv")
all_protein["analysis_group"] = "all_protein_ridge"
tables.append(all_protein)

# Leaked nominal protein model
leaked = pd.read_csv("strokecog_nominal_spearman_proteins_ridge_result.csv")
leaked["analysis_group"] = "nominal_proteins_leaked"
tables.append(leaked)

# Leakage-free selected protein model
leakage_free = pd.read_csv("strokecog_leakage_free_spearman_ridge_result.csv")
leakage_free["analysis_group"] = "nominal_proteins_leakage_free"
tables.append(leakage_free)

# Combine
master_results = pd.concat(tables, ignore_index=True, sort=False)

# Keep useful columns
cols_to_show = [
    "analysis_group", "model", "outcome_type", "estimator",
    "n_patients", "n_features", "mean_selected_features",
    "RMSE", "MAE", "Pearson_r", "Pearson_p",
    "Spearman_r", "Spearman_p"
]

cols_to_show = [c for c in cols_to_show if c in master_results.columns]

master_results_clean = master_results[cols_to_show]

display(master_results_clean)

master_results_clean.to_csv(
    "strokecog_master_model_comparison.csv",
    index=False
)

,analysis_group,model,outcome_type,estimator,n_patients,n_features,mean_selected_features,RMSE,MAE,Pearson_r,Pearson_p,Spearman_r,Spearman_p
0,clinical_only,Mean-only baseline,continuous_SIS3,NaN,85,0.0,NaN,18.207243,14.331933,-1.000000,0.000000e+00,-1.000000,0.000000e+00
1,clinical_only,Age only,continuous_SIS3,LinearRegression,85,1.0,NaN,18.279126,14.517936,-0.020515,8.521659e-01,0.042848,6.970063e-01
2,clinical_only,Time Since Stroke only,continuous_SIS3,LinearRegression,85,1.0,NaN,17.704913,14.305282,0.209987,5.374495e-02,0.087434,4.262125e-01
3,clinical_only,Age + Time Since Stroke,continuous_SIS3,LinearRegression,85,2.0,NaN,17.771052,14.415921,0.206989,5.733732e-02,0.180379,9.854148e-02
4,all_protein_ridge,All proteins only,continuous_SIS3,RidgeCV,85,1011.0,NaN,17.141062,13.733057,0.311306,3.731274e-03,0.227991,3.585406e-02
5,all_protein_ridge,All proteins + Age,continuous_SIS3,RidgeCV,85,1012.0,NaN,17.141044,13.736939,0.311375,3.723199e-03,0.226474,3.713863e-02
6,all_protein_ridge,All proteins + Time Since Stroke,continuous_SIS3,RidgeCV,85,1012.0,NaN,17.273137,13.853874,0.296049,5.941636e-03,0.225593,3.790201e-02
7,all_protein_ridge,All proteins + Age + Time Since Stroke,continuous_SIS3,RidgeCV,85,1013.0,NaN,17.224915,13.804587,0.301753,5.007106e-03,0.230145,3.409484e-02
8,nominal_proteins_leaked,Nominal Spearman proteins only,continuous_SIS3,RidgeCV,85,180.0,NaN,14.904792,11.648056,0.560316,2.459058e-08,0.508901,6.588861e-07
9,nominal_proteins_leakage_free,Leakage-free nominal Spearman proteins inside ...,continuous_SIS3,RidgeCV,85,NaN,197.870588,18.249128,14.361168,0.203446,6.183324e-02,0.132639,2.262345e-01


In [29]:
# ============================================================
# StrokeCog: Final biological interpretation summary table
# ============================================================

import pandas as pd
import numpy as np

# Load annotated protein-theme table
top_named = pd.read_csv("strokecog_top_named_proteins_with_themes.csv")

# Keep useful columns
df = top_named.copy()

# Split multi-theme proteins into long format
rows = []

for _, row in df.iterrows():
    themes = str(row["biological_theme"]).split("; ")
    for theme in themes:
        rows.append({
            "theme": theme,
            "protein": row["protein"],
            "name": row["name"],
            "panel": row["panel"],
            "spearman_rho": row["spearman_rho"],
            "p_value": row["p_value"],
            "FDR_q": row["FDR_q"],
            "direction": row["direction"]
        })

theme_long = pd.DataFrame(rows)

# Function to get top representative proteins by p-value
def representative_proteins(group, n=6):
    tmp = group.sort_values("p_value")
    names = tmp["name"].dropna().astype(str).unique().tolist()
    return ", ".join(names[:n])

# Theme-level summary
summary_rows = []

for theme, group in theme_long.groupby("theme"):
    n_total = group["name"].nunique()
    n_worse = group[group["direction"] == "higher_with_worse_mood"]["name"].nunique()
    n_better = group[group["direction"] == "higher_with_better_mood"]["name"].nunique()

    if n_worse > n_better:
        main_direction = "mostly higher with worse mood"
    elif n_better > n_worse:
        main_direction = "mostly higher with better mood"
    else:
        main_direction = "mixed direction"

    reps = representative_proteins(group, n=8)

    summary_rows.append({
        "theme": theme,
        "n_named_proteins": n_total,
        "higher_with_worse_mood": n_worse,
        "higher_with_better_mood": n_better,
        "main_direction": main_direction,
        "representative_proteins": reps
    })

theme_summary_final = pd.DataFrame(summary_rows).sort_values(
    ["higher_with_worse_mood", "n_named_proteins"],
    ascending=False
)

display(theme_summary_final)

theme_summary_final.to_csv(
    "strokecog_final_biological_theme_summary.csv",
    index=False
)

,theme,n_named_proteins,higher_with_worse_mood,higher_with_better_mood,main_direction,representative_proteins
5,other/uncurated,71,61,10,mostly higher with worse mood,"PAI, SCAMP3, DECR1, NUB1, SMAD1, VPS53, PLXDC1..."
1,immune/inflammatory,37,31,6,mostly higher with worse mood,"ITGA11, FGF2, TNFRSF10A, SRPK2, TRIM5, IRF9, C..."
0,cell-stress/cell-regulation,29,27,2,mostly higher with worse mood,"DNAJB1, TXNDC15, RNF31, PTPN1, RARRES1, SRPK2,..."
4,neuro-related,28,27,1,mostly higher with worse mood,"PVALB, RNF31, PTPN1, ABHD14B, FKBP5, EIF4B, MA..."
3,metabolism/prostaglandin,19,16,3,mostly higher with worse mood,"HPGDS, ABHD14B, SDC4, CDHR5, NAMPT, ADGRG2, CD..."
2,integrin/ECM/vascular-remodeling,11,6,5,mostly higher with worse mood,"ITGAV, ITGA11, FGF2, EGF, ANGPT1, SDC4, ITGB6,..."


In [30]:
# ============================================================
# Add interpretation and caution columns
# ============================================================

interpretation_map = {
    "immune/inflammatory": (
        "Suggests peripheral immune activation or inflammatory signaling associated with mood."
    ),
    "cell-stress/cell-regulation": (
        "Suggests systemic cellular stress, redox/protein-folding response, or altered cell regulation."
    ),
    "integrin/ECM/vascular-remodeling": (
        "Suggests vascular, extracellular matrix, adhesion, or remodeling biology."
    ),
    "metabolism/prostaglandin": (
        "Suggests metabolic, lipid mediator, or prostaglandin-related biology."
    ),
    "neuro-related": (
        "Contains proteins annotated in neuro-related panels, but plasma origin is not necessarily brain-specific."
    ),
    "other/uncurated": (
        "Proteins not classified by the simple rule-based theme system."
    )
}

theme_summary_final["interpretation"] = theme_summary_final["theme"].map(
    interpretation_map
).fillna("Biological interpretation requires manual curation.")

theme_summary_final["caution"] = (
    "Plasma proteins are systemic biomarkers and are not necessarily brain-derived. "
    "Themes are rule-based and exploratory, not formal pathway enrichment."
)

display(theme_summary_final)

theme_summary_final.to_csv(
    "strokecog_final_biological_theme_summary_with_interpretation.csv",
    index=False
)

,theme,n_named_proteins,higher_with_worse_mood,higher_with_better_mood,main_direction,representative_proteins,interpretation,caution
5,other/uncurated,71,61,10,mostly higher with worse mood,"PAI, SCAMP3, DECR1, NUB1, SMAD1, VPS53, PLXDC1...",Proteins not classified by the simple rule-bas...,Plasma proteins are systemic biomarkers and ar...
1,immune/inflammatory,37,31,6,mostly higher with worse mood,"ITGA11, FGF2, TNFRSF10A, SRPK2, TRIM5, IRF9, C...",Suggests peripheral immune activation or infla...,Plasma proteins are systemic biomarkers and ar...
0,cell-stress/cell-regulation,29,27,2,mostly higher with worse mood,"DNAJB1, TXNDC15, RNF31, PTPN1, RARRES1, SRPK2,...","Suggests systemic cellular stress, redox/prote...",Plasma proteins are systemic biomarkers and ar...
4,neuro-related,28,27,1,mostly higher with worse mood,"PVALB, RNF31, PTPN1, ABHD14B, FKBP5, EIF4B, MA...",Contains proteins annotated in neuro-related p...,Plasma proteins are systemic biomarkers and ar...
3,metabolism/prostaglandin,19,16,3,mostly higher with worse mood,"HPGDS, ABHD14B, SDC4, CDHR5, NAMPT, ADGRG2, CD...","Suggests metabolic, lipid mediator, or prostag...",Plasma proteins are systemic biomarkers and ar...
2,integrin/ECM/vascular-remodeling,11,6,5,mostly higher with worse mood,"ITGAV, ITGA11, FGF2, EGF, ANGPT1, SDC4, ITGB6,...","Suggests vascular, extracellular matrix, adhes...",Plasma proteins are systemic biomarkers and ar...
